In [ ]:
####  Uncomment this cell if you're training on GPU T4x2 because ModernBERT's Trainer Repo has had problems
## -- with Multiple GPU's.

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
from transformers import pipeline,AutoModelForSeq2SeqLM,Trainer,TrainingArguments,AutoTokenizer,AutoModelForQuestionAnswering
from datasets import load_dataset,DatasetDict
from huggingface_hub import notebook_login
import numpy as np
import pandas as pd
import torch
from collections import defaultdict

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("Hugging Face")
# Set the environment variable and log in
os.environ["Hugging Face"] = hf_token
login(token=hf_token)

In [ ]:
%pip install -q evaluate

In [ ]:
dataset=load_dataset("rajpurkar/squad")

In [ ]:
dataset

In [ ]:
dataset=DatasetDict(
    {
        "train":dataset['train'].shuffle().select(range(70000)),
        "validation":dataset['validation']
    }
)

In [ ]:
dataset['train'][0]['context']

In [ ]:
dataset['train'][0]['question']

In [ ]:
dataset['train'][0]['answers']

In [ ]:
dataset['validation'].filter(lambda x:len(x['answers']['text'])!=1)

In [ ]:
dataset['train'].filter(lambda x:len(x['answers']['text'])!=1)

In [ ]:
model_checkpoint="answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


In [ ]:
context = dataset["train"][0]["context"]
question = dataset["train"][0]["question"]


In [ ]:
inputs=tokenizer(question,context,max_length=100,stride=50,truncation='only_second',return_overflowing_tokens=True,return_offsets_mapping=True)

In [ ]:
inputs

In [ ]:
tokenizer.decode(inputs['input_ids'])

In [ ]:
max_length=384
stride=128

In [ ]:
def preprocess_data(dataset):
    questions=[q.strip() for q in dataset['question']]
    inputs=tokenizer(questions,dataset['context'],max_length=max_length,truncation="only_second",stride=stride,
                     return_overflowing_tokens=True,return_offsets_mapping=True,padding="max_length")
    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = dataset["answers"]
    start_positions = []
    end_positions = []

    for i,offset in enumerate(offset_mapping):
        sample_idx=sample_map[i]
        answer=answers[sample_idx]
        start_char=answer["answer_start"][0]
        end_char=start_char+len(answer["text"][0])
        sequence_ids=inputs.sequence_ids(i)

        idx=0
        while sequence_ids[idx]!=1:
            idx+=1
        context_start=idx
        while sequence_ids[idx]==1:
            idx+=1
        context_end=idx-1

        if offset[context_start][0]>start_char or offset[context_end][1]<end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:

            idx=context_start
            while idx<=context_end and offset[idx][0]<=start_char:
                idx+=1
            start_positions.append(idx-1)
            while idx>=context_start and offset[idx][1]>=end_char:
                idx-=1
            end_positions.append(idx+1)
    inputs["start_positions"]=start_positions
    inputs["end_positions"]=end_positions
    return inputs
        
        


In [ ]:
train_dataset=dataset['train'].map(preprocess_data,batched=True,remove_columns=dataset['train'].column_names)

In [ ]:
train_dataset

In [ ]:
dataset['validation']

In [ ]:
def preprocess_val_data(dataset):
    questions=[q.strip() for q in dataset['question']]
    inputs=tokenizer(questions,dataset['context'],max_length=max_length,truncation="only_second",stride=stride,
                     return_overflowing_tokens=True,return_offsets_mapping=True,padding="max_length")
    sample_map=inputs.pop("overflow_to_sample_mapping")      ####Mapping of each chunk to the example id
    example_ids=[]
    for i in range(len(inputs['input_ids'])):
        sample_idx=sample_map[i]
        example_ids.append(dataset['id'][sample_idx])
        sequence_ids=inputs.sequence_ids(i)
        offset=inputs['offset_mapping'][i]
        inputs['offset_mapping'][i]=[value if sequence_ids[k]==1 else None for k,value in enumerate(offset)]

    inputs['example_id']=example_ids
    return inputs
        

In [ ]:
validation_dataset = dataset["validation"].map(preprocess_val_data,batched=True,remove_columns=dataset["validation"].column_names)

In [ ]:
validation_dataset

In [ ]:
import evaluate
metric = evaluate.load("squad")


In [ ]:
from tqdm.auto import tqdm
def compute_metrics(start_logits, end_logits, features, examples):
    example_to_features = defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    n_best=20
    max_answer_length=30
    for example in tqdm(examples):
        example_id = example["id"]
        context = example["context"]
        answers = []

        # Loop through all features associated with that example
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip answers that are not fully in the context
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # Skip answers with a length that is either < 0 or > max_answer_length
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)

        # Select the answer with the best score
        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

In [ ]:
model=AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

In [ ]:
args=TrainingArguments("ModernBERT-base-finetuned-squad",eval_strategy="no",save_strategy="epoch",learning_rate=2e-5,num_train_epochs=3,weight_decay=0.01,
                       fp16=True,per_device_train_batch_size=8,per_device_eval_batch_size=2,eval_accumulation_steps=4,gradient_accumulation_steps=4,push_to_hub=True)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
trainer.push_to_hub()

In [ ]:
predictions, _, _ = trainer.predict(validation_dataset)
start_logits, end_logits = predictions
compute_metrics(start_logits, end_logits, validation_dataset, dataset["validation"])